In [1]:
# CELL 1 — Load RFM + groceries data
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

rfm = pd.read_csv('../data/clean/rfm_segments.csv')
df  = pd.read_csv('../data/clean/groceries_clean.csv')
df['order_date']       = pd.to_datetime(df['order_date'])
df['first_order_date'] = pd.to_datetime(df['first_order_date'])

print('✅ Data loaded')
print(f'RFM customers: {len(rfm):,}')
print(f'Segments: {rfm["segment"].unique().tolist()}')

✅ Data loaded
RFM customers: 3,898
Segments: ['Champion', 'Loyal', 'Potential', 'Churned', 'At-Risk']


In [2]:
# CELL 2 — Find each customer's first product category
# Map products to categories
CATEGORY_MAP = {
    'whole milk': 'Dairy',
    'yogurt': 'Dairy',
    'butter': 'Dairy',
    'cream cheese': 'Dairy',
    'domestic eggs': 'Dairy',
    'eggs': 'Dairy',
    'other vegetables': 'Fresh Produce',
    'root vegetables': 'Fresh Produce',
    'tropical fruit': 'Fresh Produce',
    'citrus fruit': 'Fresh Produce',
    'fruit/vegetable juice': 'Fresh Produce',
    'pip fruit': 'Fresh Produce',
    'rolls/buns': 'Bakery & Grains',
    'bread': 'Bakery & Grains',
    'pastry': 'Bakery & Grains',
    'rice': 'Bakery & Grains',
    'flour': 'Bakery & Grains',
    'soda': 'Beverages',
    'bottled water': 'Beverages',
    'canned beer': 'Beverages',
    'misc. beverages': 'Beverages',
    'coffee': 'Beverages',
    'bottled beer': 'Beverages',
    'sausage': 'Meat & Snacks',
    'frankfurter': 'Meat & Snacks',
    'beef': 'Meat & Snacks',
    'chicken': 'Meat & Snacks',
    'pork': 'Meat & Snacks',
    'snacks': 'Meat & Snacks',
    'salty snack': 'Meat & Snacks',
    'chips': 'Meat & Snacks'
}

# Get first order per customer
first_orders = df.sort_values('order_date').groupby('customer_id').first().reset_index()
first_orders['first_category'] = first_orders['product'].map(CATEGORY_MAP).fillna('Other')

# Merge with RFM
sankey_df = rfm.merge(first_orders[['customer_id','first_category']], on='customer_id', how='left')
sankey_df['first_category'] = sankey_df['first_category'].fillna('Other')

# Outcome: retained = Champion or Loyal, churned = Churned
sankey_df['outcome'] = sankey_df['segment'].apply(
    lambda x: 'Retained High-Value' if x == 'Champion' 
    else ('Retained' if x in ['Loyal','Potential'] 
    else 'Churned')
)

print('✅ Sankey data prepared')
print(f'\nFirst category distribution:')
print(sankey_df['first_category'].value_counts())
print(f'\nOutcome distribution:')
print(sankey_df['outcome'].value_counts())

✅ Sankey data prepared

First category distribution:
first_category
Other              1826
Fresh Produce       599
Beverages           462
Dairy               461
Bakery & Grains     308
Meat & Snacks       242
Name: count, dtype: int64

Outcome distribution:
outcome
Retained               1994
Churned                1095
Retained High-Value     809
Name: count, dtype: int64


In [3]:
# CELL 3 — Build Sankey flow data
# Flow: First Category → Segment → Outcome
categories  = sankey_df['first_category'].unique().tolist()
segments    = sankey_df['segment'].unique().tolist()
outcomes    = sankey_df['outcome'].unique().tolist()

# Create node labels
all_nodes = categories + segments + outcomes
node_idx  = {name: i for i, name in enumerate(all_nodes)}

# Build links Layer 1: Category → Segment
links_source, links_target, links_value, links_color = [], [], [], []

cat_colors = {
    'Dairy':           'rgba(220,38,38,0.6)',
    'Fresh Produce':   'rgba(29,158,117,0.6)',
    'Bakery & Grains': 'rgba(108,99,219,0.6)',
    'Beverages':       'rgba(14,165,233,0.6)',
    'Meat & Snacks':   'rgba(245,158,11,0.6)',
    'Other':           'rgba(100,116,139,0.6)'
}

seg_colors = {
    'Champion':  'rgba(220,38,38,0.6)',
    'Loyal':     'rgba(249,115,22,0.6)',
    'Potential': 'rgba(108,99,219,0.6)',
    'At-Risk':   'rgba(245,158,11,0.6)',
    'Churned':   'rgba(100,116,139,0.6)'
}

# Layer 1: Category → Segment
for cat in categories:
    for seg in segments:
        count = len(sankey_df[(sankey_df['first_category']==cat) & (sankey_df['segment']==seg)])
        if count > 0:
            links_source.append(node_idx[cat])
            links_target.append(node_idx[seg])
            links_value.append(count)
            links_color.append(cat_colors.get(cat, 'rgba(150,150,150,0.4)'))

# Layer 2: Segment → Outcome
for seg in segments:
    for out in outcomes:
        count = len(sankey_df[(sankey_df['segment']==seg) & (sankey_df['outcome']==out)])
        if count > 0:
            links_source.append(node_idx[seg])
            links_target.append(node_idx[out])
            links_value.append(count)
            links_color.append(seg_colors.get(seg, 'rgba(150,150,150,0.4)'))

print(f'✅ Sankey links built: {len(links_source)} flows')

✅ Sankey links built: 35 flows


In [4]:
# CELL 4 — THE SIGNATURE SANKEY DIAGRAM
node_colors = (
    [cat_colors.get(c, 'rgba(150,150,150,0.9)') for c in categories] +
    [seg_colors.get(s, 'rgba(150,150,150,0.9)') for s in segments] +
    ['rgba(29,158,117,0.9)', 'rgba(249,115,22,0.9)', 'rgba(100,116,139,0.9)']
)

fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=20,
        thickness=25,
        line=dict(color='#0F172A', width=1),
        label=all_nodes,
        color=node_colors,
        hovertemplate='<b>%{label}</b><br>%{value} customers<extra></extra>'
    ),
    link=dict(
        source=links_source,
        target=links_target,
        value=links_value,
        color=links_color,
        hovertemplate='%{source.label} → %{target.label}<br>%{value} customers<extra></extra>'
    )
)])

fig.update_layout(
    title=dict(
        text='🌊 Customer Journey Sankey<br><sup>First Product Category → RFM Segment → Outcome | Width = number of customers</sup>',
        font=dict(size=17, color='white')
    ),
    height=650,
    paper_bgcolor='#0F172A',
    font=dict(color='white', size=12)
)

fig.write_html('../outputs/sankey_customer_journey.html')
print('✅ SANKEY SAVED!')
print('📂 Open sankey_customer_journey.html in Chrome')
print('🔥 THIS is the chart that gets you hired')

✅ SANKEY SAVED!
📂 Open sankey_customer_journey.html in Chrome
🔥 THIS is the chart that gets you hired


In [5]:
# CELL 5 — Key Sankey Insights
print('='*60)
print('🌊 SANKEY — KEY INSIGHTS')
print('='*60)

for cat in sankey_df['first_category'].value_counts().index[:5]:
    subset = sankey_df[sankey_df['first_category'] == cat]
    champion_pct = len(subset[subset['segment']=='Champion']) / len(subset) * 100
    churned_pct  = len(subset[subset['segment']=='Churned'])  / len(subset) * 100
    total = len(subset)
    print(f'\n📦 First purchase: {cat} ({total} customers)')
    print(f'   → Became Champion: {champion_pct:.1f}%')
    print(f'   → Churned:         {churned_pct:.1f}%')

🌊 SANKEY — KEY INSIGHTS

📦 First purchase: Other (1826 customers)
   → Became Champion: 22.2%
   → Churned:         22.4%

📦 First purchase: Fresh Produce (599 customers)
   → Became Champion: 20.5%
   → Churned:         23.0%

📦 First purchase: Beverages (462 customers)
   → Became Champion: 17.1%
   → Churned:         24.9%

📦 First purchase: Dairy (461 customers)
   → Became Champion: 21.5%
   → Churned:         22.3%

📦 First purchase: Bakery & Grains (308 customers)
   → Became Champion: 20.1%
   → Churned:         21.4%


In [6]:
# CELL 6 — Save + Day 7 complete
sankey_df.to_csv('../data/clean/sankey_data.csv', index=False)

print('🎉 DAY 7 COMPLETE!')
print('='*55)
print('What you built:')
print('  ✅ Customer journey mapping')
print('  ✅ First category → Segment → Outcome flows')
print('  ✅ Sankey diagram → sankey_customer_journey.html')
print('  ✅ Category-level champion vs churn rates')
print('='*55)
print('Day 8 tomorrow: STREAMLIT DASHBOARD 🚀')
print('All your analyses come together into one live app.')
print('The most exciting day of the whole project.')

🎉 DAY 7 COMPLETE!
What you built:
  ✅ Customer journey mapping
  ✅ First category → Segment → Outcome flows
  ✅ Sankey diagram → sankey_customer_journey.html
  ✅ Category-level champion vs churn rates
Day 8 tomorrow: STREAMLIT DASHBOARD 🚀
All your analyses come together into one live app.
The most exciting day of the whole project.
